# 🎙️ Dia TTS on Google Colab

[Dia](https://github.com/nari-labs/dia) (1.6B) generates realistic dialogue from a script.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Rules: start with `[S1]`, alternate `[S1]`/`[S2]`, aim for ~5–20 s of speech, English only.
Supported non-verbals: `(laughs) (coughs) (sighs) (gasps) (clears throat)` etc. — use sparingly.

## 1. Check GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"

## 2. Install

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git librosa soundfile

## 3. Load model (first run downloads weights)

In [ ]:
import torch
from transformers import AutoProcessor, DiaForConditionalGeneration

CKPT = "nari-labs/Dia-1.6B-0626"
DTYPE = torch.float16  # T4 has weak bfloat16 support

processor = AutoProcessor.from_pretrained(CKPT)
model = DiaForConditionalGeneration.from_pretrained(CKPT, torch_dtype=DTYPE).to("cuda")
print("Model loaded.")

## 4. Generate dialogue

In [ ]:
from IPython.display import Audio, display

def generate(text, out_path="output.mp3", seed=42, audio=None,
             guidance_scale=3.0, temperature=1.8, top_p=0.90, top_k=45):
    torch.manual_seed(seed)  # same seed -> same voices
    kwargs = {"audio": audio} if audio is not None else {}
    inputs = processor(text=[text], padding=True, return_tensors="pt", **kwargs).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=3072,
                             guidance_scale=guidance_scale, temperature=temperature,
                             top_p=top_p, top_k=top_k)
    out = processor.batch_decode(out, **kwargs)
    processor.save_audio(out, out_path)
    display(Audio(out_path))
    return out_path

script = (
    "[S1] Dia is an open weights text to dialogue model. "
    "[S2] You get full control over scripts and voices. "
    "[S1] Wow. Amazing. (laughs) "
    "[S2] Try it now on Github or Hugging Face."
)
generate(script, "example.mp3", seed=42)

## 5. (Optional) Voice cloning

Upload a **5–10 s** clip and write its exact transcript with speaker tags.
The transcript goes **before** your new script. Only the new part is spoken in the output.

In [ ]:
from google.colab import files
import librosa

up = files.upload()                      # choose your audio file
path = next(iter(up))
audio, _ = librosa.load(path, sr=44100)  # Dia expects 44.1 kHz

transcript = "[S1] Write the exact words spoken in your uploaded clip here."
new_text   = " [S1] And this is the new sentence I want in the cloned voice."

generate(transcript + new_text, "cloned.mp3", seed=42, audio=audio)

## 6. 🖥️ Full web UI (all features in one place)

Run this cell to get a Gradio app with a public link:
- script editor with `[S1]`/`[S2]` and non-verbal tag buttons
- voice cloning (audio + transcript)
- seed, CFG, temperature, top-p/top-k, max tokens
- automatic chunking for long scripts

Needs sections 1–3 to be run first. Stop the cell to shut the UI down.

In [ ]:
import re, random
import numpy as np, soundfile as sf, librosa
import gradio as gr

SR = 44100
NONVERBALS = ["(laughs)","(clears throat)","(sighs)","(gasps)","(coughs)","(singing)","(sings)",
              "(mumbles)","(beep)","(groans)","(sniffs)","(claps)","(screams)","(inhales)",
              "(exhales)","(applause)","(burps)","(humming)","(sneezes)","(chuckle)","(whistles)"]

def split_turns(text):
    return [t.strip() for t in re.split(r"(?=\[S[12]\])", text) if t.strip()]

def synth(text, seed, audio, gen):
    torch.manual_seed(int(seed))
    kw = {"audio": audio} if audio is not None else {}
    inputs = processor(text=[text], padding=True, return_tensors="pt", **kw).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, **gen)
    dec = processor.batch_decode(out, **kw)
    return np.asarray(dec[0], dtype=np.float32).squeeze()

def run(script, seed, rand_seed, guidance, temperature, top_p, top_k, max_tokens,
        clone_file, clone_text, turns_per_chunk):
    script = (script or "").strip()
    if not script.startswith("[S1]"):
        raise gr.Error("Script must start with [S1] and alternate [S1]/[S2].")
    if rand_seed:
        seed = random.randint(0, 2**31 - 1)

    audio = None
    if clone_file:
        if not clone_text or not clone_text.strip().startswith("[S"):
            raise gr.Error("Voice cloning needs the clip's transcript with [S1]/[S2] tags.")
        audio, _ = librosa.load(clone_file, sr=SR)

    gen = dict(max_new_tokens=int(max_tokens), guidance_scale=guidance,
               temperature=temperature, top_p=top_p, top_k=int(top_k))

    turns = split_turns(script)
    n = int(turns_per_chunk)
    chunks = [" ".join(turns[i:i+n]) for i in range(0, len(turns), n)]

    pieces = []
    for c in chunks:
        text = (clone_text.strip() + " " + c) if audio is not None else c
        pieces.append(synth(text, seed, audio, gen))
        pieces.append(np.zeros(int(0.15 * SR), dtype=np.float32))  # short pause
    wav = np.concatenate(pieces)
    sf.write("ui_output.wav", wav, SR)
    return "ui_output.wav", int(seed)

def add_tag(script, tag):
    return (script or "") + (" " if script else "") + tag

with gr.Blocks(title="Dia TTS") as demo:
    gr.Markdown("## 🎙️ Dia — text to dialogue\nStart with `[S1]`, alternate `[S1]`/`[S2]`. English only.")
    script = gr.Textbox(label="Script", lines=8,
        value="[S1] Dia is an open weights text to dialogue model. [S2] You get full control over scripts and voices. [S1] Wow. Amazing. (laughs) [S2] Try it now.")
    with gr.Row():
        tag = gr.Dropdown(NONVERBALS, value="(laughs)", label="Non-verbal tag")
        gr.Button("➕ Insert tag").click(add_tag, [script, tag], script)
        gr.Button("[S1]").click(lambda s: add_tag(s, "[S1]"), script, script)
        gr.Button("[S2]").click(lambda s: add_tag(s, "[S2]"), script, script)

    with gr.Accordion("🎭 Voice cloning (optional)", open=False):
        clone_file = gr.Audio(type="filepath", label="Reference audio (5–10 s)")
        clone_text = gr.Textbox(label="Exact transcript of the reference audio, with tags",
                                placeholder="[S1] Words spoken in the clip. [S2] ...")
    with gr.Accordion("⚙️ Advanced", open=False):
        seed = gr.Number(value=42, precision=0, label="Seed")
        rand_seed = gr.Checkbox(False, label="Random seed")
        guidance = gr.Slider(1.0, 6.0, 3.0, step=0.1, label="CFG scale")
        temperature = gr.Slider(0.5, 2.5, 1.8, step=0.05, label="Temperature")
        top_p = gr.Slider(0.5, 1.0, 0.90, step=0.01, label="Top-p")
        top_k = gr.Slider(1, 100, 45, step=1, label="Top-k")
        max_tokens = gr.Slider(512, 4096, 3072, step=64, label="Max new tokens (~86 tokens = 1 s)")
        turns_per_chunk = gr.Slider(1, 10, 4, step=1, label="Speaker turns per chunk (long scripts are split)")

    out_audio = gr.Audio(label="Output", type="filepath")
    used_seed = gr.Number(label="Seed used", interactive=False)
    gr.Button("🚀 Generate", variant="primary").click(
        run, [script, seed, rand_seed, guidance, temperature, top_p, top_k, max_tokens,
              clone_file, clone_text, turns_per_chunk], [out_audio, used_seed])

demo.launch(share=True, debug=True)

## 7. Download results


In [ ]:
from google.colab import files
files.download("example.mp3")
# files.download("cloned.mp3")

## Troubleshooting
- **CUDA out of memory:** Runtime → Restart session, then run again.
- **Weird artifacts:** fewer non-verbal tags, shorter text, try another `seed`.
- **Too fast speech:** script too long — split it into shorter chunks.
- **Voice cloning errors:** transformers' audio-prompt API may change; check `hf.py` in the official repo.